# Characterization of loop dynamics in kinases

We present a workflow to discover protein conformational features associated with kinase loop rearrangments. The purpose of this notebook is to describe the necessary steps adopted in our study. Implementations of the described steps are included as `.py` files within the folder `workflow`.

## Table of contents

This modelling pipeline is subdivided in the following sections:

1. [Dataset creation](#1)
   1. [The data](#11)
   2. [Data download](#12)
   3. [Kinase taxonomy](#13)
2. [Dataset curation](#2)
   1. [Extracting protein chains](#21)
   2. [Extracting ligands](#22)
   3. [Filtering for activation loop](#23)
   4. [Conformational classification](#24)
   5. [Structural conservation](#25)
   6. [Reconstructing small loop segments](#26)
   7. [Coarse-graining activation loops](#27)
   8. [Cα interpolation](#28)



The overall pipeline, implemented in the sections hereafter, is represented according to the following schematic.

![State of the workflow](images/AllPipeline.png)

In this notebook, we will be focusing on the first step: dataset curation.

![State of the workflow](images/DatasetCuration.png)

To get started, let's load some packages!

In [ ]:
# File and system operations
import os
import sys
import subprocess
from glob import glob
import pickle
import shutil

# Data processing
import pandas as pd
import numpy as np
import mdtraj as md

# Network and parallel processing
import requests
import time
import multiprocessing
import concurrent.futures

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# custom utility functions and class
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs, copy_motif_filtered_datasets
from workflow.utilities import PDBDownloader

# 1. Dataset creation <a id="1"></a>

In this section we will parse the Protein Data Bank (PDB) for kinase structures and download them into a dataset.

## 1.1 The data <a id="11"></a>
Our approach involves searching for protein homologs to the reference sequence: a BRAF kinase (PDB code: 6UAN).

BRAF is a key part of the MAPK/ERK pathway. This pathway relays signals from outside the cell, like growth factors binding to receptor tyrosine kinases, to control cell growth, division, survival, and differentiation.

The active site sits in a cleft between the small N‑terminal lobe and larger C‑terminal lobe of the kinase domain. 
ATP binds in a pocket on the N‑lobe side of the kinase domain, at the P-binding loop. 
The protein substrate, mainly MEK kinase, contacts a broad surface on the C‑lobe of BRAF. 
Activation loop and αC helix interact with residues in the binding site, regulating activation. 

![State of the workflow](images/BRAFSlide1.png)

When creating our dataset, we take as structural reference the BRAF structure since we are familiar with its typical regulatory role during phosphorilation. We query the InterPro database online at https://www.ebi.ac.uk/interpro/ to find structures in the PDB that match the protein kinase-like domain family. InterPro is a database that classifies protein sequences into families and predicts the presence of domains and important sites.

Our query input is the BRAF sequence and we filter for structures that are part of the "Protein kinase-like domain superfamily" (IPR011009) and that are included in the PDB.

## 1.2 Data download <a id="12"></a>
Here we download the structures output from the InterPro query.

Let's start by writing all PDB codes to a list.

In [ ]:
structure_path = 'structure-matching-IPR011009.tsv'
pdb_data = pd.read_csv(structure_path, sep = "\t", header=0, engine='python')
pdb_data['Accession'] = pdb_data['Accession'].str.upper()
pdb_ids = pdb_data['Accession'].tolist()

The class `PDBDownloader` enables carrying out multi-threaded PDB download. It uses up to 2 CPU cores. We now download the PDB structures listed above.

In [ ]:
downloader = PDBDownloader()
downloader.parallel_download(pdb_ids, "Results/InterProPDBs") 

We can now check how many structures from the InterPro query were actually downloaded.

In [ ]:
folder_path = "Results/InterProPDBs"
file_names = [os.path.splitext(f)[0] for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
pdb_raw = pd.DataFrame({"PDBs": file_names})

pdb_data['Downloaded'] = pdb_data['Accession'].str.upper().isin(pdb_raw['PDBs']).map({True: True, False: False})

counts = pdb_data['Downloaded'].value_counts().to_dict()
print(f"Downloaded: {counts[True]}, Failed: {counts[False]}")

We can save the names of failed PDB downloads for future reference.

In [ ]:
fail_list = pdb_data[pdb_data['Downloaded']==False]
fail_list.to_csv('fail_list.csv')

## 1.3 Kinase taxonomy <a id="13"></a>
We seek to annotate our dataset with kinase family, species, and class information.


The class `KinaseGroupLabeller` enables extracting metadata from UniProt to investigate what kinase families and which species are represented in our dataset.

In [ ]:
from workflow.kinaseGroupLabelling import KinaseGroupLabeller

lab = KinaseGroupLabeller()

# Annotate each extracted PDB *chain* in your dataset directory
# (one row per <PDB>_<CHAIN>.pdb file)
dataset_dir = "Results/activation_segments/unaligned/"  # change if needed
annot = lab.annotate_dataset_chains_with_kinome(
    dataset_dir,
    output_csv="Results/kinase_annotation_chains.csv",
)

display(annot.head())


We now use the plotting method `plot_distribution_bars()` to visualise the parsed metadata as histograms.

In [ ]:
figs = lab.plot_chain_annotation_distributions(
    annot_path="Results/kinase_annotation_chains.csv",
)
annot = figs["annot"]


Our dataset includes only Kinase-like structures as expected.

# 2. Dataset curation <a id="2"></a>
In this section we will curate our kinase dataset for input into conformational analysis.

## 2.1 Extracting protein chains <a id="21"></a>
Here we extract only the protein chains containing a kinase domain from our database of downloaded PDB structures.

We utilise the class `PDBChainExtractor()` to write to PDB files the coordinates of chains indicated by InterPro query output. 

In [ ]:
from workflow.pdb_chain_extractor import PDBChainExtractor

# Create an instance of the class
chain_extractor = PDBChainExtractor()

# --- Dataset 1: protein-only chains (no ligands) ---
chain_extractor.extract_chains_parallel(
    pdb_data,
    target_dir='Results/activation_segments/unaligned/',
    max_workers=None,
    include_ligands=False,
)

Let's make sure that the number of chains corresponds to at least the same amount of files downloaded.

In [ ]:
pdb_directory = 'Results/activation_segments/unaligned/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.2 Extracting ligands <a id="22"></a>
Here we extract both the kinase protein chains and any bound ligand from our database of downloaded PDB structures.

We utilise the class `PDBChainExtractor()` to write to PDB files the atomic coordinates of chains and any ligand within 5 Å. 

In [ ]:
from workflow.pdb_chain_extractor import PDBChainExtractor

# Create an instance of the class
chain_extractor = PDBChainExtractor()

# --- Dataset 2: protein chains + nearby ligands/cofactors/ions (for KinCore ligand detection) ---
# This keeps non-protein atoms within `ligand_distance` Å of the chain.
# It also writes `<output>.ligands.tsv` inventories alongside each extracted PDB.
chain_extractor.extract_chains_parallel(
    pdb_data,
    target_dir='Results/activation_segments/unaligned+ligands/',
    max_workers=None,
    include_ligands=True,
    ligand_distance=5.0,
    keep_waters=False,
    write_ligand_inventory=True,
)

Let's now report on which PDB structures contain ligands of interest.

In [ ]:
from workflow.pdb_chain_extractor import PDBChainExtractor

chain_extractor = PDBChainExtractor()

df_lig = chain_extractor.report_ligands_in_directory(
    "Results/activation_segments/unaligned+ligands/",
    output_csv="Results/activation_segments/unaligned+ligands_ligand_report.csv",
    exclude_waters=True,
    exclude_amino_acids=True,
    print_examples=50,
)

Let's make sure that the number of structures matches the number of chains in the previous step.

In [ ]:
pdb_directory = 'Results/activation_segments/unaligned+ligands/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.3 Filtering for activation loop  <a id="23"></a>
Here we exclude all kinase domains that do not have the characteristic conserved residue motifs DFG and APE that delimit the activation loop.

We utilise the method `copy_filtered_pdbs()` to extract amino acid sequences from the structures in our dataset and exclude those not containing DFG and APE. We will do this both for the chain-only and chain with ligand datasets.

In [ ]:
valid_pdbs, invalid_pdbs = copy_motif_filtered_datasets(
    source_dir_protein="Results/activation_segments/unaligned/",
    target_dir_protein="Results/activation_segments/motif_filtered/",
    source_dir_ligands="Results/activation_segments/unaligned+ligands/",
    target_dir_ligands="Results/activation_segments/motif_filtered+ligands/",
)


Let's check how many kinase domains we are left with in both datasets.

In [ ]:
pdb_directory = 'Results/activation_segments/motif_filtered/'
pdb_directory2 = 'Results/activation_segments/motif_filtered+ligands/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directories '{pdb_directory}' and '{pdb_directory2}'.")

## 2.4 Conformational classification  <a id="24"></a>
Here we investigate the conformational diversity of our kinase domains by applying the classification developed by the Dunbrack's group.

The class `DunbrackWorkflow` enables performing the conformational classification using the `KinCore` software.

**When KinCore fails**, structures are marked with `'failed'` status. This happens when:
- KinCore cannot find the DFG or C-helix motifs
- Structure has missing residues in critical regions
- Non-standard kinase fold
- Structure quality issues

In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

paths = DunbrackWorkflow.build_output_paths(output_dir="Results/dunbrack_assignments")
DUNBRACK_OUTPUT_DIR = paths["output_dir"]
KINCORE_ASSIGNMENTS_CSV = paths["assignments_csv"]
TRUE_LIGAND_CSV = paths["true_ligand_csv"]
CONFORMATION_PLOT_PNG = paths["conformation_plot_png"]

FORCE_KINCORE = False

_ = DunbrackWorkflow.ensure_assignments_cached(
    input_dir="Results/activation_segments/motif_filtered+ligands/",
    output_dir=DUNBRACK_OUTPUT_DIR,
    kincore_dir="/home/marmatt/Documents/Kincore-standalone",
    assignments_filename="kinase_conformation_assignments.csv",
    force=FORCE_KINCORE,
)


Let's now print some information about the KinCore analysis.

In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

ligand_report_df = DunbrackWorkflow.report_from_assignments_csv(
    assignments_csv=KINCORE_ASSIGNMENTS_CSV,
    true_ligand_csv=TRUE_LIGAND_CSV,
)


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

type1_sel_df = DunbrackWorkflow.summarize_type1_chains(
    pdb_dir="Results/activation_segments/motif_filtered+ligands",
    true_ligand_csv=TRUE_LIGAND_CSV,
)

display(type1_sel_df.head(50))
print("(showing first 50 rows)")


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

atp_manifest_df = DunbrackWorkflow.copy_atp_analogue_structures(
    pdb_dir="Results/activation_segments/motif_filtered+ligands",
    true_ligand_csv=TRUE_LIGAND_CSV,
    combined_out_dir="Results/activation_segments/ATP+analogues",
)

display(atp_manifest_df.head(50))
print("(showing first 50 rows)")


Let's now visualise the metadata extracted from `KinCore`.

In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

_ = DunbrackWorkflow.show_or_plot_conformation_distribution(
    assignments_csv=KINCORE_ASSIGNMENTS_CSV,
    output_png=CONFORMATION_PLOT_PNG,
    show=True,
    print_dunbrack_summary=True,
)


## 2.5 Structural conservation  <a id="25"></a>
Here we will be investigating which residues of our reference BRAF kinase are structurally conserved across the collected dataset.

We choose to assess structure conservation using a novel multiple structure alignment algorithm: FoldMason. It uses the structural alphabet from Foldseek to represent 3D structures as sequences, enabling fast comparison between large structure sets. The class `AlignmentFoldMason` is implemented for this purpose.

In [ ]:
from workflow.align_FoldMason import AlignmentFoldMason

# Initialize
aligner = AlignmentFoldMason(log_file="multiple_alignment_foldmason.log")

# Single multi-structure FoldMason run (optionally anchors with the template first)
# NOTE: use report_mode=2 to generate `msa.json`, which we then parse to export `msa_tmscore.csv`.
aligner.process_foldmason_alignment_multi(
    pdb_path="Results/activation_segments/motif_filtered/",
    target_dir="Results/activation_segments/multi_aligned_foldmason/",
    template_pdb="6UAN_chainD.pdb",  # omit if you don't want to include a template
    out_name="msa",                  # output prefix
    report_mode=2                    # 0: none, 1: HTML report, 2: JSON report (enables TM-score CSV export)
)

Let's check that the number of structurally aligned files corresponds to the same number of files filtered for activation segment in the previous subsection.

In [ ]:
pdb_directory = 'Results/activation_segments/multi_aligned_foldmason/'
pdb_count = count_pdb_files(pdb_directory, recursive=True)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.") 

We will be showing structure conservation with respect to the BRAF reference sequence. We have written the `analyse_alignment()` class to load the multi-structure alignment, calculate conservation at each BRAF residue position and select residues that fall within a certain conservation threshold (70%). We visualise this analysis as a histogram.

**This class creates the `conservation` variable** that is used later in the feature selection workflow.

In [ ]:
from workflow.analyse_alignment_foldmason import analyse_alignment

analyser = analyse_alignment()
conservation_run = analyser.run_multi_alignment_conservation_analysis(
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    reference_name="6UAN_chainD",
    reference_residues=braf_res(),
    conservation_threshold=0.70,
    output_plot="Results/multi_alignment_foldMason_conservation.png",
    output_csv="Results/conserved_residues_70percent.csv",
    show_plot=True,
)

conserved_df = conservation_run["conserved_df"]


In order to assess the validity of our approach we can visualise how FoldMason aligns sequences by running the method `visualise_sequence_alignment()`.

In [ ]:
from workflow.analyse_alignment_foldmason import visualise_sequence_alignment

# Create visualizer instance
visualizer = visualise_sequence_alignment()

# Generate HTML for 3Di alignment
di_stats = visualizer.generate_multi_alignment_html(
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    output_file="Results/multi_alignment_3di.html",
    reference_name="6UAN_chainD"
)

print(f"3Di alignment: {di_stats}")